# Ejercicio de clase: Predicción del precio de vivienda con Decision Tree Regressor

**Dataset**: California Housing Prices — https://www.kaggle.com/datasets/camnugent/california-housing-prices

En el ejemplo de clase (`decision_tree_example.ipynb`) usamos un **Decision Tree Classifier** para predecir
si un paciente sobrevivía o no a una sepsis. Esa era una tarea de **clasificación**: la variable que
queríamos predecir (`hospital_outcome`) solo podía tomar un número limitado de valores (0 o 1, "vive" o
"muere").

En este ejercicio vamos a resolver un problema distinto: **regresión**. Vamos a predecir el **precio
mediano de una vivienda** (`median_house_value`) en un bloque censal de California, a partir de
características como la ubicación, la cantidad de habitaciones o el ingreso medio de sus habitantes.

> **Diferencia clave**: en clasificación el modelo predice una *categoría* (una clase). En regresión el
> modelo predice un *número real* (puede tomar, en principio, cualquier valor dentro de un rango continuo).
> Esto tiene consecuencias importantes: no podemos usar métricas como *precision*, *recall* o *F1*, porque
> esas métricas comparan clases exactas. En su lugar usaremos métricas que midan **qué tan lejos** está la
> predicción del valor real, como el **MAE (Mean Absolute Error)**, que veremos más adelante.

Para este ejercicio usarán `DecisionTreeRegressor` en lugar de `DecisionTreeClassifier`.

### Antes de empezar

1. Descarguen el dataset desde Kaggle: https://www.kaggle.com/datasets/camnugent/california-housing-prices
2. El archivo se llama `housing.csv`. Colóquenlo dentro de la carpeta `raw/` de este proyecto
   (la misma carpeta `decision_tree/raw/` donde está el dataset de sepsis).
3. Sigan cada sección en orden. Cada sección tiene:
   - Una breve explicación de qué vamos a hacer y por qué.
   - Un **reto**: ustedes deben escribir el código (no está resuelto).
   - **Preguntas de análisis**: respóndanlas en una celda de markdown justo debajo de su código.


### Importar librerías

Igual que en el ejemplo de clase, empezamos importando las librerías que vamos a necesitar. Noten dos
diferencias frente al ejemplo de clasificación:

- Usamos `DecisionTreeRegressor` en lugar de `DecisionTreeClassifier`.
- Usamos `mean_absolute_error` en lugar de `precision_score`, `recall_score`, `f1_score`.


In [ ]:
!pip install pandas
!pip install scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import mean_absolute_error


## Paso 1 — Cargar los datos

**Reto**: Carguen el archivo `housing.csv` (carpeta `raw/`) en un DataFrame de pandas llamado `df`, tal
como hicimos en el ejemplo de clase con `pd.read_csv(...)`. Luego muestren las primeras filas del
DataFrame para confirmar que se cargó correctamente.


In [ ]:
df = pd.read_csv("housing.csv")

In [ ]:
print(df.shape)

In [ ]:
df.head()

**Preguntas de análisis**
1. ¿Cuántas filas y cuántas columnas tiene el dataset? (pista: `df.shape`)

   (20640, 10)

2. ¿Qué representa cada fila del dataset? ¿Es una vivienda individual o algo distinto?

   cada fila representa un bloque de California que es una pequeña unidad geográfica agrupada por el censo de EE.UU.

3. Observando los nombres de las columnas, ¿cuál creen que es la variable que vamos a predecir (el
   *target*)?

   median_house_value — es la única columna que representa directamente un "resultado" (el precio), mientras que todas las demás (ubicación, edad de las viviendas, ingresos, cercanía al mar, etc.) son características que describen el bloque y que tiene sentido usar para explicar o predecir ese precio.


## Paso 2 — Identificación inicial de los datos

Antes de tocar cualquier dato, siempre debemos entender con qué estamos trabajando: cuántas columnas hay,
qué tipo de dato tiene cada una, si hay valores nulos, y cuál es el rango de valores de cada variable.

**Reto**: Usando lo que ya conocen de pandas, respondan (con código) estas tres preguntas:
1. ¿Qué tipo de dato (`dtype`) tiene cada columna? ¿Hay alguna columna categórica (texto)?

   df.dtypes

2. ¿Cuáles son las estadísticas básicas (media, desviación estándar, mínimo, máximo, etc.) de las
   columnas numéricas?

   df.describe()
3. ¿Hay valores nulos (faltantes) en el dataset? ¿En qué columna(s)?

   df.isnull().sum()

Pistas de los métodos que necesitan (ya los usamos, en otra forma, en el ejemplo de clase): `.info()`,
`.describe()`, `.isnull()`.


In [ ]:
df.dtypes

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
# Ver las categorías de ocean_proximity
df["ocean_proximity"].unique()

In [ ]:
# Ver el precio máximo
df["median_house_value"].max()

In [ ]:
# Ver el precio minimo
df["median_house_value"].min()

In [ ]:
# Contar cuántas casas tienen el precio máximo
df[df["median_house_value"] == df["median_house_value"].max()].shape[0]

**Preguntas de análisis**
1. ¿Cuál es la única columna categórica (de texto) del dataset? ¿Qué valores puede tomar?

   ocean_proximity ['NEAR BAY', '<1H OCEAN', 'INLAND', 'NEAR OCEAN', 'ISLAND']

2. ¿Qué columna tiene valores nulos? ¿Cuántas filas están afectadas, aproximadamente qué porcentaje del
   total representa?

   total_bedrooms  207 de 2,640 datos que es el 1%

3. Miren el `min` y el `max` de `median_house_value`. ¿Les parece un rango razonable para el precio de una
   vivienda? ¿Notan algo raro en el valor máximo? (pista: busquen cuántas filas tienen exactamente ese
   valor máximo).

   El mínimo parece razonable para una vivienda económica, pero el máximo es sospechoso: 965 filas comparten exactamente el mismo valor máximo de $500,001. Eso no es una coincidencia — es la señal clásica de que el dataset original topó (cap) artificialmente los precios en $500,001
   
4. Comparen el rango de `median_income` con el de `total_rooms`. ¿Están en escalas muy distintas? ¿Creen
   que eso sería un problema para un Decision Tree? (piensen en cómo el árbol elige los cortes: ¿necesita
   que las variables estén en la misma escala, como sí lo necesitan otros modelos?)

   sí, están en escalas muy distintas — median_income va de 0.4999 a 15.0001 (decenas de miles de dólares, expresado en una unidad reescalada), mientras que total_rooms va de 2 a 39,320 (habitaciones totales en el bloque censal). Son órdenes de magnitud diferentes.


## Paso 3 — Procesamiento de datos

Por ahora, para mantener el ejercicio simple, vamos a trabajar **solo con variables numéricas**. Más
adelante en el curso aprenderemos técnicas para incorporar variables categóricas (como *one-hot encoding*),
pero hoy las vamos a descartar.

**Reto**:
1. Eliminen del DataFrame la(s) columna(s) categórica(s) que identificaron en el paso anterior.
2. Decidan qué hacer con los valores nulos que encontraron (por ejemplo, eliminar esas filas) y
   apliquen esa decisión. Un Decision Tree Regressor de scikit-learn no puede entrenarse si quedan
   valores `NaN` en los datos.
3. Confirmen, con código, que ya no quedan columnas categóricas ni valores nulos.


In [ ]:
df_num = df.drop(columns=["ocean_proximity"])

In [ ]:
df_num = df_num.dropna()

In [ ]:
print("Valores nulos:")
print(df_num.isnull().sum())

print("\nTipos de datos:")
print(df_num.dtypes)

**Preguntas de análisis**
1. ¿Qué información de las viviendas estamos perdiendo al eliminar la columna categórica? ¿Creen que esa
   información podría ser útil para predecir el precio? ¿Por qué?

   ocena_proximity Esa información sí podría ser muy útil para predecir el precio: en California, las viviendas cerca de la costa valen  más que las que están lejos. Al eliminar esta columna, el modelo pierde una señal potencialmente fuerte y tiene que intentar "adivinar" esa cercanía indirectamente a partir de longitude y latitude — que sí quedan en el dataset, pero de forma mucho menos directa e interpretable que una categoría explícita.

2. Si eliminaron filas con nulos, ¿cuántas filas quedaron en total? ¿Qué porcentaje del dataset original
   se perdió?

   se perdieron 207 filas que es el 1% no es significativo

3. ¿Qué otra estrategia (distinta a eliminar las filas) existe para tratar valores nulos? ¿Por qué hoy
   optamos por la más simple?

   En vez de eliminar las filas con nulos, se podría hacer imputación: rellenar los valores faltantes de total_bedrooms con algún valor estimado

   como solo representa 1% del dataset, el impacto de perder esas filas es casi nulo


## Paso 4 — Definir variables predictoras (X) y variable objetivo (y), y dividir los datos

Igual que en el ejemplo de clase, necesitamos separar:
- **X**: las columnas que el modelo usará para predecir (todas menos el precio).
- **y**: la columna que queremos predecir (`median_house_value`).

Y luego dividir ambas en un conjunto de **entrenamiento** y uno de **prueba**, usando
`train_test_split`, tal como hicimos con los datos de sepsis.

**Reto**:
1. Construyan `X` (todas las columnas numéricas excepto `median_house_value`) y `y`
   (`median_house_value`).
2. Usen `train_test_split` para crear `X_train`, `X_test`, `y_train`, `y_test`. Usen un `test_size` de
   0.2 y `random_state=0` para que los resultados sean reproducibles.


In [ ]:
# variables predictoras.
X = df_num.drop(columns=["median_house_value"])

# variable objetivo.
y = df_num["median_house_value"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=0
)

**Preguntas de análisis**
1. ¿Cuántas filas quedaron en `X_train` y cuántas en `X_test`?

   El dataset queda con 20.433 filas despues de eliminar las 207 filas con valores nulos y como utilizamos test_size = 0.2, entonces aproximadamente usamos el 80% para entrenamiento y el 20% para prueba, por ende , x_train tiene 16.346 filas y x_test tiene 4.087 filas.

2. ¿Por qué es importante evaluar el modelo en datos que **no** usó para entrenar (`X_test`, `y_test`)?
   ¿Qué pasaría si evaluáramos únicamente sobre `X_train`?

   Es importante utilizar datos que el modelo no usó en el entrenamiento para comprobar si realmente puede hacer buenas predicciones con datos nuevos.

   Si se evalúa únicamente sobre X_train, podríamos obtener un resultado demasiado bueno porque el modelo ya aprendió esos datos. Esto no nos dejaría saber si el modelo funciona bien con datos que no conoce.

3. En el ejemplo de clase balanceamos las clases con SMOTE antes de dividir los datos. Aquí no lo hicimos.
   ¿Por qué SMOTE (que genera ejemplos sintéticos de una *clase* minoritaria) no tiene sentido en un
   problema de regresión, donde no hay clases sino un valor continuo?

   SMOTE se utiliza en problemas de clasificación para generar nuevos ejemplos de una clase minoritaria y así equilibrar las clases. En este caso estamos haciendo regresión, por lo que no existen clases como "vive" o "muere", sino un valor continuo que es median_house_value, que representa el precio de la casa. Por lo que SMOTE no es apropiado para este ejercicio.


## Paso 5 — Entrenar el Decision Tree Regressor

**Reto**: Entrenen un `DecisionTreeRegressor` (con `random_state=0`) usando `X_train` y `y_train`, de la
misma forma en que entrenaron el `DecisionTreeClassifier` en el ejemplo de clase.


In [ ]:
reg = DecisionTreeRegressor(random_state=0)

reg.fit(X_train, y_train)

**Preguntas de análisis**
1. En clasificación, cada hoja del árbol predice una clase (por ejemplo, "vive" o "muere"). En regresión,
   ¿qué creen que predice cada hoja del árbol? (pista: piensen en los valores de `y` que caen en esa
   hoja durante el entrenamiento).

   Cada hoja de este árbol predice un valor numérico para las viviendas que llegan a esa hoja, en este caso, la predicción corresponde al valor promedio de median_house_value de los datos de entrenamiento que quedaron dentro de esa hoja.

2. ¿Qué criterio usa por defecto `DecisionTreeRegressor` para decidir dónde hacer cada corte, en lugar del
   *gini* o *entropy* que se usan en clasificación? (revisen la documentación del parámetro `criterion`).

   El criterio que utiliza por defecto es squared_error, o sea el error cuadrático. El árbol busca realizar cortes que reduzcan la variación de los valores de y dentro de los grupos resultantes.

## Paso 6 — Evaluar el modelo con MAE

### ¿Qué es el MAE (Mean Absolute Error)?

El **MAE** es el promedio de la diferencia absoluta entre el valor real y el valor predicho:

$$MAE = \frac{1}{n}\sum_{i=1}^{n} |y_i - \hat{y}_i|$$

A diferencia de precision/recall/F1 (que solo tienen sentido cuando comparamos clases), el MAE funciona
sobre valores numéricos continuos y nos dice, **en promedio, cuánto se equivoca el modelo, en las mismas
unidades que la variable objetivo**. En nuestro caso, como `median_house_value` está en dólares, un
MAE de, por ejemplo, 40000 significa que en promedio el modelo se equivoca en $40,000 dólares al predecir
el precio de una vivienda.

Un MAE más bajo es mejor. Pero un mismo valor de MAE puede ser "bueno" o "malo" dependiendo de la escala
de la variable que estamos prediciendo — por eso siempre hay que compararlo contra algo (por ejemplo,
el precio promedio de las viviendas).

**Reto**:
1. Usen el modelo entrenado para predecir sobre `X_train` y calculen el MAE comparando esas predicciones
   con `y_train`.
2. Hagan lo mismo sobre `X_test` con `y_test`.
3. Comparen ambos valores.


In [ ]:
y_train_pred = reg.predict(X_train)

mae_train = mean_absolute_error(
    y_train,
    y_train_pred
)

print("MAE (train):", mae_train)

In [ ]:
y_test_pred = reg.predict(X_test)

mae_test = mean_absolute_error(
    y_test,
    y_test_pred
)

print("MAE (test):", mae_test)

**Preguntas de análisis**
1. ¿El MAE de entrenamiento es mayor, menor o similar al MAE de prueba? ¿Qué les dice eso sobre qué tan
   bien "memorizó" el árbol los datos de entrenamiento?

   El MAE de entrenamiento es menor que el MAE de prueba, como se puede ver el MAE de entrenamiento es 0.0, mientras que el MAE de prueba es aproximadamente 43.557 dólares.

2. Calculen el precio promedio (`.mean()`) de `median_house_value` en todo el dataset. Comparando ese
   promedio con el MAE de prueba, ¿el error del modelo les parece grande o pequeño en proporción al precio
   típico de una vivienda?

   


3. En el ejemplo de clase, un árbol sin restricciones (`fully grown`) mostraba señales de sobreajuste
   (*overfitting*) al compararlo con un árbol más simple (`max_depth=3`). Según los MAE de train y test
   que obtuvieron, ¿creen que este árbol también está sobreajustado? ¿Por qué?


## Reto adicional (opcional) — ¿Se puede mejorar el árbol?

En el ejemplo de clase, limitar la profundidad del árbol (`max_depth=3`) cambió el comportamiento del
modelo. Prueben lo mismo aquí:

1. Entrenen un segundo `DecisionTreeRegressor`, esta vez fijando `max_depth` (prueben con distintos
   valores, por ejemplo 3, 5, 10).
2. Calculen el MAE de train y de test para cada valor de `max_depth`.
3. Grafiquen (opcional) el MAE de train y de test contra `max_depth`, como una curva.

**Preguntas de análisis**
1. ¿Qué le pasa al MAE de entrenamiento a medida que aumentan `max_depth`? ¿Y al de prueba?
2. ¿Existe un valor de `max_depth` donde el MAE de prueba deja de mejorar (o empeora)? ¿Qué relación tiene
   eso con el concepto de *overfitting* que vimos en el ejemplo de clase?
3. Miren `reg.feature_importances_` del árbol entrenado. ¿Cuáles son las 2 o 3 variables más importantes
   para predecir el precio de la vivienda? ¿Tiene sentido con lo que ustedes esperarían intuitivamente?


In [ ]:
# Espacio libre para el reto adicional
